<style>
:root { --green:#087f5b; --mint:#e9f7f1; --gold:#d99a00; --gold-soft:#fff7d6; --blue-soft:#edf5ff; --ink:#17352c; }
.project-hero { padding:28px; border-radius:18px; background:linear-gradient(135deg,var(--mint),var(--gold-soft)); border-left:7px solid var(--green); color:var(--ink); }
.project-hero h1 { margin:0 0 10px; color:var(--green); }
.project-badge { display:inline-block; margin-top:10px; padding:5px 10px; border-radius:999px; background:var(--green); color:white; font-weight:600; }
.jp-MarkdownOutput h2 { margin-top:26px; padding:10px 14px; border-left:5px solid var(--green); border-radius:8px; background:var(--mint); color:var(--ink); }
.jp-CodeCell { border-left:4px solid var(--gold); border-radius:8px; background:#fffdf5; }
.project-note,.project-result,.project-warning { margin:12px 0; padding:12px 15px; border-radius:10px; }
.project-note { background:var(--blue-soft); border-left:4px solid #3478b8; }
.project-result { background:var(--mint); border-left:4px solid var(--green); }
.project-warning { background:var(--gold-soft); border-left:4px solid var(--gold); }
</style>


<div class="project-hero">
  <h1>Projet NLP - Reconnaissance d'entités nommées en wolof</h1>
  <p><strong>Mon objectif :</strong> reconnaître les personnes, lieux, organisations et dates dans un texte wolof.</p>
  <p>Je construis un CRF simple, mesurable et facile à expliquer.</p>
  <span class="project-badge">MasakhaNER · Wolof · BIO · CRF</span>
</div>

<div class="project-note"><strong>Ma démarche :</strong> comprendre les données, vérifier BIO, préparer X/y, entraîner, sélectionner sur dev et évaluer une seule fois sur test.</div>


## 1. Importer les outils nécessaires

Le notebook utilise les fonctions courtes et testées du dossier `src`.

In [1]:
from collections import Counter
from pathlib import Path

from src.analyze_data import find_bio_errors, load_conll
from src.dataset import prepare_split
from src.error_analysis import analyze_entity_errors
from src.evaluate import evaluate_entities
from src.select_crf import select_best_crf
from src.train_crf import predictions_are_aligned, train_crf

## 2. Charger train et dev

- `train` sert à apprendre ;
- `dev` sert à régler le modèle ;
- `test` reste fermé jusqu'à l'évaluation finale.

In [2]:
DATA_DIR = Path("data/raw")

# Une ligne vide du fichier CoNLL sépare deux phrases.
train_sentences = load_conll(DATA_DIR / "train.txt")
dev_sentences = load_conll(DATA_DIR / "dev.txt")
for name, sentences in [
    ("train", train_sentences),
    ("dev", dev_sentences),
]:
    token_count = sum(len(sentence.tokens) for sentence in sentences)
    print(f"{name:5s} : {len(sentences):4d} phrases | {token_count:5d} tokens")

train : 1871 phrases | 36805 tokens
dev   :  267 phrases |  4384 tokens


## 3. Vérifier les labels et le format BIO

`B-TYPE` commence une entité, `I-TYPE` la continue et `O` signifie hors entité.

In [3]:
# Compter les labels permet de voir le déséquilibre du corpus.
train_label_counts = Counter(
    label
    for sentence in train_sentences
    for label in sentence.labels
)

print("Labels de train :", dict(sorted(train_label_counts.items())))
print("Erreurs BIO dans train :", len(find_bio_errors(train_sentences)))
print("Erreurs BIO dans dev   :", len(find_bio_errors(dev_sentences)))

Labels de train : {'B-DATE': 130, 'B-LOC': 566, 'B-ORG': 179, 'B-PER': 522, 'I-DATE': 170, 'I-LOC': 21, 'I-ORG': 129, 'I-PER': 440, 'O': 34648}
Erreurs BIO dans train : 0
Erreurs BIO dans dev   : 0


### 3.1 Quantifier le déséquilibre et les entités

Ces calculs transforment l'observation en preuves chiffrées.

In [4]:
# Une entité BIO commence toujours par B-.
entity_counts = Counter(
    label[2:]
    for sentence in train_sentences
    for label in sentence.labels
    if label.startswith("B-")
)
sentence_lengths = [len(sentence.tokens) for sentence in train_sentences]
token_total = sum(train_label_counts.values())
o_ratio = train_label_counts["O"] / token_total

print("Entités de train :", dict(entity_counts))
print(f"Part du label O  : {o_ratio:.2%}")
print(
    "Longueur des phrases - min :", min(sentence_lengths),
    "| moyenne :", round(sum(sentence_lengths) / len(sentence_lengths), 2),
    "| max :", max(sentence_lengths),
)

Entités de train : {'PER': 522, 'ORG': 179, 'LOC': 566, 'DATE': 130}
Part du label O  : 94.14%
Longueur des phrases - min : 2 | moyenne : 19.67 | max : 83


Le label `O` est majoritaire. L'accuracy seule serait donc trompeuse : nous utiliserons le F1 au niveau des entités.

## 4. Construire X et y

- `X` contient les caractéristiques des tokens ;
- `y` contient les labels BIO ;
- les positions doivent rester alignées.

In [5]:
# Chaque token de X conserve le label situé à la même position dans y.
X_train, y_train = prepare_split(train_sentences)
X_dev, y_dev = prepare_split(dev_sentences)

print("Nombre de phrases dans X_train :", len(X_train))
print("Nombre de phrases dans y_train :", len(y_train))
print("Premier token observé :", train_sentences[0].tokens[0])
print("Ses caractéristiques :", X_train[0][0])
print("Son label attendu     :", y_train[0][0])

Nombre de phrases dans X_train : 1871
Nombre de phrases dans y_train : 1871
Premier token observé : SAFIYETU
Ses caractéristiques : {'bias': 1.0, 'word.lower': 'safiyetu', 'word.prefix1': 's', 'word.prefix2': 'sa', 'word.suffix2': 'tu', 'word.suffix3': 'etu', 'word.shape': 'XXXXXXXX', 'word.isupper': True, 'word.istitle': False, 'word.isdigit': False, 'BOS': True, '+1:word.lower': 'béey', '+1:word.istitle': False, '+1:word.isupper': True}
Son label attendu     : B-PER


## 5. Entraîner le CRF

Le CRF apprend les associations caractéristiques-labels et les transitions entre labels successifs.

In [6]:
# Configuration : L-BFGS + régularisations L1/L2 + toutes les transitions.
# fit() apprend uniquement avec train ; dev et test ne participent pas à l'apprentissage.
crf_model = train_crf(X_train, y_train)
print("Entraînement terminé.")

Entraînement terminé.


### 5.1 Vérifier ce que le CRF a réellement appris

Les poids positifs élevés montrent les transitions BIO favorisées.

In [7]:
# transition_features_ contient un poids appris pour chaque paire de labels.
learned_transitions = sorted(
    crf_model.transition_features_.items(),
    key=lambda item: item[1],
    reverse=True,
)

print("Transitions les plus favorisées :")
for (label_before, label_after), weight in learned_transitions[:8]:
    print(f"{label_before:7s} -> {label_after:7s} : {weight:.3f}")

print("\nNombre de poids de caractéristiques :", len(crf_model.state_features_))
print("Nombre de poids de transitions     :", len(crf_model.transition_features_))

Transitions les plus favorisées :
I-ORG   -> I-ORG   : 5.071
B-ORG   -> I-ORG   : 4.953
B-PER   -> I-PER   : 4.941
B-DATE  -> I-DATE  : 4.444
I-DATE  -> I-DATE  : 4.303
I-LOC   -> I-LOC   : 3.141
I-PER   -> I-PER   : 2.948
B-LOC   -> I-LOC   : 2.862

Nombre de poids de caractéristiques : 3824
Nombre de poids de transitions     : 62


## 6. Produire les prédictions sur dev

Nous contrôlons la forme et l'alignement des sorties avant de calculer les métriques.

In [8]:
# predict() retourne un label pour chaque token de chaque phrase de dev.
dev_predictions = crf_model.predict(X_dev)

print("Phrases prédites :", len(dev_predictions))
print("Prédictions alignées :", predictions_are_aligned(dev_predictions, y_dev))

# Comparer une courte phrase sans utiliser dev pour réentraîner le modèle.
print("\nTokens     :", dev_sentences[0].tokens[:12])
print("Référence  :", y_dev[0][:12])
print("Prédiction :", dev_predictions[0][:12])

Phrases prédites : 267
Prédictions alignées : True

Tokens     : ('Ndekete', ',', 'da', 'cee', 'am', 'ñu', 'xër', 'ci', 'diine', 'ji', 'ba', 'mu')
Référence  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Prédiction : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DATE', 'O', 'O', 'O']


## 7. Évaluer les entités sur dev

`seqeval` reconstruit les entités BIO. Le mode strict exige le bon type et les bonnes limites.

In [9]:
# Calcul strict IOB2 au niveau des entités, et non des tokens isolés.
dev_metrics = evaluate_entities(y_dev, dev_predictions)

print(f"Précision : {dev_metrics['precision']:.4f}")
print(f"Rappel    : {dev_metrics['recall']:.4f}")
print(f"F1 strict : {dev_metrics['f1']:.4f}")
print("\nDétail par type d'entité :")
print(dev_metrics["report"])

Précision : 0.7453
Rappel    : 0.7248
F1 strict : 0.7349

Détail par type d'entité :
              precision    recall  f1-score   support

        DATE     0.5000    0.8333    0.6250         6
         LOC     0.8511    0.6780    0.7547        59
         ORG     0.7778    0.6364    0.7000        11
         PER     0.6750    0.8182    0.7397        33

   micro avg     0.7453    0.7248    0.7349       109
   macro avg     0.7010    0.7415    0.7049       109
weighted avg     0.7710    0.7248    0.7375       109



Le F1 micro est la métrique principale. Le rapport par type distingue PER, LOC, ORG et DATE.

## 8. Analyser les erreurs sur dev

L'analyse distingue entité inventée, manquée, mauvais type et mauvaises limites.

In [10]:
# Regrouper les erreurs réelles sans modifier le modèle ni utiliser test.
dev_errors = []
for sentence, true_labels, predicted_labels in zip(
    dev_sentences, y_dev, dev_predictions
):
    for error in analyze_entity_errors(
        sentence.tokens, true_labels, predicted_labels
    ):
        dev_errors.append({"sentence": " ".join(sentence.tokens), **error})

error_counts = Counter(error["kind"] for error in dev_errors)
print("Nombre d'erreurs par catégorie :", dict(error_counts))

# Afficher quelques cas concrets pour formuler des hypothèses.
for error in dev_errors[:5]:
    print("\nType       :", error["kind"])
    print("Référence  :", error["expected"])
    print("Prédiction :", error["predicted"])
    print("Phrase     :", error["sentence"])

Nombre d'erreurs par catégorie : {'invented': 14, 'missed': 16, 'wrong_boundaries': 9, 'wrong_type': 4}

Type       : invented
Référence  : aucune entité
Prédiction : diine · DATE
Phrase     : Ndekete , da cee am ñu xër ci diine ji ba mu ëpp , naan réew mi mbas du fi dug ndax sunu maam yu baax yi fi tëdd .

Type       : missed
Référence  : DIC · ORG
Prédiction : aucune entité
Phrase     : Waaye , bi leen DIC wooloo , ku nekk ci ñoom dellu na ginnaaw .

Type       : wrong_boundaries
Référence  : Michel Aulas · PER
Prédiction : Michel Aulas Njiitu Olympique Lyonnais · PER
Phrase     : Jean - Michel Aulas Njiitu Olympique Lyonnais .

Type       : missed
Référence  : Jean · PER
Prédiction : aucune entité
Phrase     : Jean - Michel Aulas Njiitu Olympique Lyonnais .

Type       : wrong_boundaries
Référence  : Marseille · LOC
Prédiction : Meeri Marseille · PER
Phrase     : Ndege , ci atum 2014 , ci wutaakon yi doon sàkku Meeri Marseille bi la bokkoon .


Sur `dev`, les entités manquées et inventées sont les erreurs les plus fréquentes. Les exemples sont inspectés avant d'affirmer une cause.

## 9. Comparer quelques configurations sur dev

Seuls `c1` et `c2` changent. La cellule suivante entraîne réellement les quatre modèles.

In [11]:
# Cette cellule entraîne réellement les quatre CRF sur train.
# Chaque modèle est évalué sur dev avec le même F1 strict.
best_name, selected_crf, comparison_results = select_best_crf(
    X_train, y_train, X_dev, y_dev
)

for result in comparison_results:
    print(
        f"{result['configuration']:11s} | c1={result['c1']:.2f} "
        f"c2={result['c2']:.2f} | P={result['precision']:.4f} "
        f"R={result['recall']:.4f} | F1={result['f1']:.4f}"
    )

best_result = max(comparison_results, key=lambda row: row["f1"])
print("\nConfiguration retenue :", best_name)
print("Meilleur F1 dev       :", round(best_result["f1"], 4))

initial     | c1=0.10 c2=0.10 | P=0.7453 R=0.7248 | F1=0.7349
l1_forte    | c1=0.50 c2=0.10 | P=0.7524 R=0.7248 | F1=0.7383
l2_forte    | c1=0.10 c2=0.50 | P=0.7917 R=0.6972 | F1=0.7415
equilibree  | c1=0.05 c2=0.20 | P=0.7843 R=0.7339 | F1=0.7583

Configuration retenue : equilibree
Meilleur F1 dev       : 0.7583


<div class="project-result"><b>Choix final :</b> c1 = 0,05 et c2 = 0,20, car cette configuration obtient le meilleur F1 dev : 0,7583.</div>

## 10. Résultat final sur test

Après la sélection sur `dev`, la configuration est figée. `test` est ensuite ouvert une seule fois.

In [12]:
# Lire les résultats enregistrés sans réutiliser test pour choisir le modèle.
import json

metadata_path = Path("models/crf_wolof_metadata.json")
final_result = json.loads(metadata_path.read_text(encoding="utf-8"))

print("Configuration :", {"c1": final_result["c1"], "c2": final_result["c2"]})
print(f"Précision test : {final_result['test_precision']:.4f}")
print(f"Rappel test    : {final_result['test_recall']:.4f}")
print(f"F1 strict test : {final_result['test_f1']:.4f}")
print("\nDétail par type :")
print(final_result["test_report"])

Configuration : {'c1': 0.05, 'c2': 0.2}
Précision test : 0.7749
Rappel test    : 0.5781
F1 strict test : 0.6622

Détail par type :
              precision    recall  f1-score   support

        DATE     0.5238    0.1571    0.2418        70
         LOC     0.8859    0.7725    0.8253       211
         ORG     0.6129    0.3455    0.4419        55
         PER     0.7055    0.5852    0.6398       176

   micro avg     0.7749    0.5781    0.6622       512
   macro avg     0.6820    0.4651    0.5372       512
weighted avg     0.7450    0.5781    0.6406       512



<div class="project-warning"><b>Lecture honnête :</b> le F1 passe de 0,7583 sur dev à 0,6622 sur test. LOC généralise bien, tandis que DATE et ORG ont un rappel faible.</div>

## 11. Utiliser le modèle sauvegardé

Cette cellule montre la chaîne complète : texte, tokens, labels BIO et entités.

In [13]:
from src.predict import predict_text

# On peut remplacer cette phrase pour tester le modèle.
demo = predict_text("Maki Sàll dem Dakar")
print("Tokens  :", demo["tokens"])
print("Labels  :", demo["labels"])
print("Entités :", demo["entities"])

Tokens  : ['Maki', 'Sàll', 'dem', 'Dakar']
Labels  : ['B-PER', 'I-PER', 'O', 'O']
Entités : [{'text': 'Maki Sàll', 'type': 'PER'}]


## 12. Comparer le CRF à une baseline réellement testée

La baseline mémorise le label le plus fréquent de chaque mot de `train`. Un mot inconnu reçoit `O`.

In [14]:
from src.baseline import MemorizationBaseline

# Même protocole : apprentissage sur train, comparaison sur dev.
baseline = MemorizationBaseline().fit(train_sentences)
baseline_predictions = baseline.predict(dev_sentences)
baseline_metrics = evaluate_entities(y_dev, baseline_predictions)

print(f"Baseline - précision : {baseline_metrics['precision']:.4f}")
print(f"Baseline - rappel    : {baseline_metrics['recall']:.4f}")
print(f"Baseline - F1        : {baseline_metrics['f1']:.4f}")
print("CRF final - F1 dev  : 0.7583")

Baseline - précision : 0.8289
Baseline - rappel    : 0.5780
Baseline - F1        : 0.6811
CRF final - F1 dev  : 0.7583


| Modèle sur dev | Précision | Rappel | F1 strict |
|---|---:|---:|---:|
| Mémorisation | 0,8289 | 0,5780 | 0,6811 |
| CRF sélectionné | 0,7843 | 0,7339 | **0,7583** |

Le CRF possède un meilleur rappel grâce au contexte local et aux transitions BIO.

## 13. Architecture que je dois savoir expliquer

Le CRF combine :

1. **émissions** : compatibilité entre les caractéristiques d'un token et un label ;
2. **transitions** : compatibilité entre deux labels voisins ;
3. **régularisation** : `c1` et `c2` limitent le surapprentissage.

`fit()` apprend les poids sur `train`, puis le décodage choisit la meilleure séquence globale.

## 14. Difficultés, apprentissages et portée

- **Difficultés :** données déséquilibrées, entités rares et mots inconnus.
- **Apprentissages :** préserver BIO, séparer train/dev/test et évaluer les entités complètes.
- **Application bonus :** Streamlit compare le CRF brut à un mode assisté clairement séparé.
- **Suite :** comparer avec AfroXLM-R et ajouter davantage de données wolof.

## Conclusion

Le CRF est simple, rapide et interprétable. Il dépasse la mémorisation, mais comprend moins bien le contexte profond qu'un transformeur préentraîné.